# 04 — TFT (Temporal Fusion Transformer)

Notebook ini berdiri sendiri: jalankan `00_Preprocessing.ipynb` sekali dulu
(datanya diambil dari BigQuery), lalu Run All di sini untuk train + evaluasi
+ upload model TFT saja.

Input-nya tabel `preprocessed_data_daily` + `preprocessing_metadata_daily`,
hasil `00_Preprocessing.ipynb` (dengan `GRANULARITY = "daily"`) atau
`make preprocess` (granularity `daily` di `config/preprocessing.yaml`).

In [ ]:
from google.colab import auth
auth.authenticate_user()

from google.cloud import bigquery

import io
import json
import pickle
import numpy as np
import pandas as pd

PROJECT_ID = "thesis-506605"
DATASET    = "tables"
LOCATION   = "US"

PREPROCESSED_TABLE = f"{PROJECT_ID}.{DATASET}.preprocessed_data_daily"
METADATA_TABLE     = f"{PROJECT_ID}.{DATASET}.preprocessing_metadata_daily"

client = bigquery.Client(project=PROJECT_ID, location=LOCATION)

# --- Metadata: kolom kalender/stasiun/fitur + mapping nama kolom BigQuery ---
meta_row = client.query(f"""
    SELECT payload
    FROM `{METADATA_TABLE}`
    ORDER BY saved_at DESC
    LIMIT 1
""").to_dataframe().iloc[0]
metadata = json.loads(meta_row["payload"])
CALENDAR_COLS = metadata["CALENDAR_COLS"]
station_cols  = metadata["station_cols"]
feature_cols  = metadata["feature_cols"]
column_map    = metadata["column_map"]                 # station -> kolom BigQuery
reverse_column_map = {v: k for k, v in column_map.items()}
N_FEATURES = len(feature_cols)


# --- df_train / df_val / df_test dari tabel preprocessed_data --------------
def load_split(split):
    query = f"""
        SELECT *
        FROM `{PREPROCESSED_TABLE}`
        WHERE split = @split
        ORDER BY datetime
    """
    job_config = bigquery.QueryJobConfig(query_parameters=[
        bigquery.ScalarQueryParameter("split", "STRING", split),
    ])
    df = client.query(query, job_config=job_config).to_dataframe()
    df = df.set_index("datetime").rename(columns=reverse_column_map)
    return df[feature_cols]


df_train = load_split("train")
df_val   = load_split("val")
df_test  = load_split("test")

print(f"Stations : {station_cols}")
print(f"Features : {N_FEATURES}")
print(f"Train/Val/Test: {len(df_train):,} / {len(df_val):,} / {len(df_test):,} rows")

## Konfigurasi BigQuery — Hasil Model

Tabel hasil untuk model ini (semua di dataset `tables`):

- `tft_val_metrics_daily` — metrik validasi per (station, horizon, metric).
- `tft_weights_daily` — bobot terlatih per (stasiun, horizon), disimpan sebagai `BYTES`.
- `tft_predictions_daily` — prediksi test-set per (station, horizon, datetime).
- `tft_metrics_daily` — metrik test-set per (station, horizon, metric).

Bobot ditulis ke BigQuery segera setelah training (bukan ke disk), sehingga
langkah test-set evaluation di bawah bisa memuat ulang bobot dari BigQuery
saja bila objek model tidak lagi ada di memori (mis. notebook dibuka ulang).

In [ ]:
MODEL_NAME = "tft"

VAL_METRICS_TABLE = f"{PROJECT_ID}.{DATASET}.{MODEL_NAME}_val_metrics_daily"
WEIGHTS_TABLE     = f"{PROJECT_ID}.{DATASET}.{MODEL_NAME}_weights_daily"
PREDICTIONS_TABLE = f"{PROJECT_ID}.{DATASET}.{MODEL_NAME}_predictions_daily"
METRICS_TABLE     = f"{PROJECT_ID}.{DATASET}.{MODEL_NAME}_metrics_daily"

WRITE_DISPOSITION = "WRITE_TRUNCATE"      # atau "WRITE_APPEND"


def load_df(df, table_id, schema):
    job_config = bigquery.LoadJobConfig(
        schema=schema,
        write_disposition=WRITE_DISPOSITION,
    )
    job = client.load_table_from_dataframe(df, table_id, job_config=job_config)
    job.result()   # tunggu selesai
    tbl = client.get_table(table_id)
    print(f"  {table_id}: {tbl.num_rows:,} rows")


print("Val metrics ->", VAL_METRICS_TABLE)
print("Weights     ->", WEIGHTS_TABLE)
print("Predictions ->", PREDICTIONS_TABLE)
print("Metrics     ->", METRICS_TABLE)

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import mean_squared_error, mean_absolute_error

DEVICE     = torch.device("cuda" if torch.cuda.is_available() else "cpu")
LOOKBACK   = 3       # hari
HORIZONS   = [1, 3, 7]   # hari ke depan
BATCH_SIZE = 64
EPOCHS     = 50
LR         = 1e-3
PATIENCE   = 7       # early stopping

# Hyperparameter TFT (selaras dgn LSTM/GRU)
TFT_HIDDEN      = 64
TFT_HEADS       = 4
TFT_DROPOUT     = 0.2
TFT_LSTM_LAYERS = 1

print(f"Device     : {DEVICE}")
print(f"Lookback   : {LOOKBACK} hari")
print(f"TFT hidden={TFT_HIDDEN} | heads={TFT_HEADS} | "
      f"dropout={TFT_DROPOUT} | lstm_layers={TFT_LSTM_LAYERS}")
print(f"Models     : {len(station_cols)} stasiun \u00d7 {len(HORIZONS)} horizon = {len(station_cols)*len(HORIZONS)}")

In [ ]:
class TMADataset(Dataset):
    def __init__(self, df, target_station, feature_cols, lookback, horizon):
        self.X, self.y = [], []
        data   = df[feature_cols].values.astype(np.float32)
        target = df[target_station].values.astype(np.float32)

        for i in range(lookback, len(data) - horizon + 1):
            self.X.append(data[i - lookback:i])          # (lookback, n_features)
            self.y.append(target[i + horizon - 1])        # scalar t+h

        self.X = torch.tensor(np.array(self.X), dtype=torch.float32)
        self.y = torch.tensor(np.array(self.y), dtype=torch.float32)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

## TFT (Temporal Fusion Transformer)

Implementasi TFT mengikuti arsitektur Lim et al. (2021), memakai `TMADataset`,
`train_epoch`/`eval_epoch`/`get_predictions`, `compute_metrics`, serta
konvensi per-stasiun × per-horizon yang sama seperti LSTM/GRU.

Komponen utama:

- **Variable Selection Network (VSN)** — memberi bobot per-fitur (TMA antar
  stasiun + fitur kalender) di setiap timestep, sehingga model dapat menekan
  fitur yang tidak relevan alih-alih memperlakukan semua input setara seperti
  LSTM/GRU biasa. Ini juga menjadi sumber *interpretability* (importance fitur).
- **Gated Residual Network (GRN)** — blok non-linear dengan gating (GLU) +
  koneksi residual + LayerNorm; memungkinkan informasi melewati transformasi
  hanya bila diperlukan, menstabilkan pelatihan pada sinyal TMA yang *spiky*.
- **LSTM encoder** — memproses seluruh jendela *lookback* (3 hari) untuk
  memberi ringkasan konteks lokal (local processing) sebelum atensi.
- **Interpretable Multi-Head Attention** — menangkap dependensi jangka panjang
  antar timestep di dalam lookback.
- **Static enrichment** — pada setup ini tidak ada kovariat statis eksternal,
  sehingga enrichment memakai konteks agregat dari encoder (tetap disertakan
  agar arsitektur lengkap dan mudah diperluas).

Karena tugasnya adalah *direct single-step forecasting* (prediksi skalar
`t+h`), head TFT mereduksi keluaran atensi pada posisi terakhir menjadi satu
nilai — identik dengan cara LSTM/GRU mengambil `out[:, -1, :]`.

In [ ]:
# --- TFT building blocks -----------------------------------------------------
import torch.nn.functional as F


class GatedLinearUnit(nn.Module):
    """GLU: mengatur seberapa besar informasi diteruskan (gating)."""
    def __init__(self, input_size, hidden_size=None, dropout=0.1):
        super().__init__()
        hidden_size = hidden_size or input_size
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(input_size, hidden_size * 2)
        self.hidden_size = hidden_size

    def forward(self, x):
        x = self.dropout(x)
        x = self.fc(x)
        return F.glu(x, dim=-1)   # split terakhir jadi (value, gate)


class GatedResidualNetwork(nn.Module):
    """
    GRN(x) = LayerNorm(x + GLU(ELU(W1 x + b1) -> W2))
    Blok inti TFT: transformasi non-linear yang bisa "dilewati" via gating.
    """
    def __init__(self, input_size, hidden_size, output_size=None, dropout=0.1,
                 context_size=None):
        super().__init__()
        output_size = output_size or input_size
        self.input_size = input_size
        self.output_size = output_size

        self.fc1 = nn.Linear(input_size, hidden_size)
        self.context = (nn.Linear(context_size, hidden_size, bias=False)
                        if context_size is not None else None)
        self.fc2 = nn.Linear(hidden_size, hidden_size)
        self.glu = GatedLinearUnit(hidden_size, output_size, dropout)
        self.layer_norm = nn.LayerNorm(output_size)

        # proyeksi skip bila dimensi input != output
        self.skip = (nn.Linear(input_size, output_size)
                     if input_size != output_size else None)

    def forward(self, x, context=None):
        residual = x if self.skip is None else self.skip(x)
        h = self.fc1(x)
        if self.context is not None and context is not None:
            h = h + self.context(context)
        h = F.elu(h)
        h = self.fc2(h)
        h = self.glu(h)
        return self.layer_norm(h + residual)


class VariableSelectionNetwork(nn.Module):
    """
    Memberi bobot softmax per-fitur di tiap timestep, lalu menggabungkan
    representasi tiap fitur (masing-masing lewat GRN) dengan bobot tsb.
    Input : (batch, time, n_features)
    Output: (batch, time, hidden_size), plus bobot (batch, time, n_features)
    """
    def __init__(self, n_features, hidden_size, dropout=0.1):
        super().__init__()
        self.n_features = n_features
        self.hidden_size = hidden_size

        # GRN untuk menghitung bobot seleksi (dari seluruh fitur flatten)
        self.flattened_grn = GatedResidualNetwork(
            n_features, hidden_size, n_features, dropout
        )
        # GRN per-fitur (skalar -> hidden)
        self.per_feature_grn = nn.ModuleList([
            GatedResidualNetwork(1, hidden_size, hidden_size, dropout)
            for _ in range(n_features)
        ])

    def forward(self, x):
        # x: (B, T, F)
        weights = self.flattened_grn(x)                 # (B, T, F)
        weights = torch.softmax(weights, dim=-1).unsqueeze(-1)  # (B, T, F, 1)

        transformed = torch.stack([
            self.per_feature_grn[i](x[..., i:i + 1])    # (B, T, hidden)
            for i in range(self.n_features)
        ], dim=-2)                                       # (B, T, F, hidden)

        combined = (transformed * weights).sum(dim=-2)   # (B, T, hidden)
        return combined, weights.squeeze(-1)


class InterpretableMultiHeadAttention(nn.Module):
    """
    Multi-head attention dgn value bersama antar head (versi interpretable TFT).
    """
    def __init__(self, hidden_size, n_heads, dropout=0.1):
        super().__init__()
        assert hidden_size % n_heads == 0
        self.n_heads = n_heads
        self.d_head = hidden_size // n_heads

        self.q = nn.Linear(hidden_size, hidden_size)
        self.k = nn.Linear(hidden_size, hidden_size)
        self.v = nn.Linear(hidden_size, self.d_head)     # value dibagi antar head
        self.out = nn.Linear(self.d_head, hidden_size)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B, T, _ = x.shape
        q = self.q(x).view(B, T, self.n_heads, self.d_head).transpose(1, 2)
        k = self.k(x).view(B, T, self.n_heads, self.d_head).transpose(1, 2)
        v = self.v(x).unsqueeze(1)                        # (B,1,T,d_head)

        scores = (q @ k.transpose(-2, -1)) / (self.d_head ** 0.5)
        attn = torch.softmax(scores, dim=-1)
        attn = self.dropout(attn)

        ctx = attn @ v                                    # (B,H,T,d_head)
        ctx = ctx.mean(dim=1)                             # rata-rata antar head
        return self.out(ctx), attn


class TFTForecaster(nn.Module):
    """
    Temporal Fusion Transformer untuk direct single-step forecasting.
    Signature forward identik dgn LSTM/GRUForecaster: x (B, LOOKBACK, F) -> (B,)
    """
    def __init__(self, n_features, hidden_size=64, n_heads=4,
                 dropout=0.2, lstm_layers=1):
        super().__init__()
        self.vsn = VariableSelectionNetwork(n_features, hidden_size, dropout)

        self.lstm_encoder = nn.LSTM(
            input_size=hidden_size, hidden_size=hidden_size,
            num_layers=lstm_layers, batch_first=True,
            dropout=dropout if lstm_layers > 1 else 0.0,
        )
        # gate + norm setelah encoder (post-LSTM gating TFT)
        self.post_lstm_glu = GatedLinearUnit(hidden_size, hidden_size, dropout)
        self.post_lstm_norm = nn.LayerNorm(hidden_size)

        # static enrichment memakai konteks agregat encoder
        self.enrichment = GatedResidualNetwork(
            hidden_size, hidden_size, hidden_size, dropout,
            context_size=hidden_size,
        )
        self.attention = InterpretableMultiHeadAttention(
            hidden_size, n_heads, dropout
        )
        self.post_attn_glu = GatedLinearUnit(hidden_size, hidden_size, dropout)
        self.post_attn_norm = nn.LayerNorm(hidden_size)

        self.position_wise = GatedResidualNetwork(
            hidden_size, hidden_size, hidden_size, dropout
        )
        self.head = nn.Linear(hidden_size, 1)

    def forward(self, x, return_weights=False):
        # 1) variable selection
        selected, var_weights = self.vsn(x)              # (B,T,H)

        # 2) local processing (LSTM encoder)
        lstm_out, (h_n, _) = self.lstm_encoder(selected)
        lstm_out = self.post_lstm_norm(
            self.post_lstm_glu(lstm_out) + selected      # residual skip
        )

        # 3) static enrichment (konteks = hidden terakhir encoder)
        context = h_n[-1].unsqueeze(1).expand(-1, lstm_out.size(1), -1)
        enriched = self.enrichment(lstm_out, context)    # (B,T,H)

        # 4) temporal self-attention
        attn_out, attn_weights = self.attention(enriched)
        attn_out = self.post_attn_norm(
            self.post_attn_glu(attn_out) + enriched
        )

        # 5) position-wise feed-forward + head pada timestep terakhir
        out = self.position_wise(attn_out)
        pred = self.head(out[:, -1, :]).squeeze(-1)       # (B,)

        if return_weights:
            return pred, var_weights, attn_weights
        return pred

In [ ]:
def train_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss = 0
    for X, y in loader:
        X, y = X.to(DEVICE), y.to(DEVICE)
        optimizer.zero_grad()
        pred = model(X)
        loss = criterion(pred, y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(y)
    return total_loss / len(loader.dataset)


def eval_epoch(model, loader, criterion):
    model.eval()
    total_loss = 0
    with torch.no_grad():
        for X, y in loader:
            X, y = X.to(DEVICE), y.to(DEVICE)
            pred = model(X)
            total_loss += criterion(pred, y).item() * len(y)
    return total_loss / len(loader.dataset)


def get_predictions(model, loader):
    model.eval()
    preds, trues = [], []
    with torch.no_grad():
        for X, y in loader:
            preds.append(model(X.to(DEVICE)).cpu().numpy())
            trues.append(y.numpy())
    return np.concatenate(preds), np.concatenate(trues)


def compute_metrics(y_true, y_pred):
    mask = ~np.isnan(y_true) & ~np.isnan(y_pred)
    y_true, y_pred = y_true[mask], y_pred[mask]

    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae  = mean_absolute_error(y_true, y_pred)

    # sMAPE \u2014 symmetric, aman untuk nilai mendekati 0
    smape = np.mean(
        2 * np.abs(y_true - y_pred) / (np.abs(y_true) + np.abs(y_pred) + 1e-8)
    ) * 100

    # NSE \u2014 Nash-Sutcliffe Efficiency
    nse = 1 - (np.sum((y_true - y_pred) ** 2) /
               np.sum((y_true - np.mean(y_true)) ** 2 + 1e-8))

    return {"RMSE": rmse, "MAE": mae, "sMAPE": smape, "NSE": nse}

In [ ]:
tft_results = {}
tft_models  = {}

criterion = nn.MSELoss()

for target in station_cols:
    print(f"\n{'='*50}")
    print(f"Target: {target}")
    print(f"{'='*50}")

    tft_models[target]  = {}
    tft_results[target] = {}

    for h in HORIZONS:
        train_ds = TMADataset(df_train, target, feature_cols, LOOKBACK, h)
        val_ds   = TMADataset(df_val,   target, feature_cols, LOOKBACK, h)

        train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
        val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False)

        model = TFTForecaster(
            n_features=N_FEATURES,
            hidden_size=TFT_HIDDEN,
            n_heads=TFT_HEADS,
            dropout=TFT_DROPOUT,
            lstm_layers=TFT_LSTM_LAYERS,
        ).to(DEVICE)
        optimizer = torch.optim.Adam(model.parameters(), lr=LR)

        best_val_loss = float("inf")
        best_state    = None
        patience_ctr  = 0

        for epoch in range(EPOCHS):
            train_loss = train_epoch(model, train_loader, optimizer, criterion)
            val_loss   = eval_epoch(model, val_loader, criterion)

            if val_loss < best_val_loss:
                best_val_loss = val_loss
                best_state    = {k: v.clone() for k, v in model.state_dict().items()}
                patience_ctr  = 0
            else:
                patience_ctr += 1
                if patience_ctr >= PATIENCE:
                    print(f"  h+{h:>2}d | early stop epoch {epoch+1:>3} | "
                          f"val_loss={best_val_loss:.6f}")
                    break

        model.load_state_dict(best_state)
        tft_models[target][h] = model

        y_pred, y_true = get_predictions(model, val_loader)
        metrics = compute_metrics(y_true, y_pred)
        tft_results[target][f"h+{h}"] = metrics

        print(f"  h+{h:>2}d \u2192 RMSE={metrics['RMSE']:.4f} | "
              f"MAE={metrics['MAE']:.4f} | "
              f"sMAPE={metrics['sMAPE']:.2f}% | "
              f"NSE={metrics['NSE']:.4f}")

In [ ]:
weights_schema = [
    bigquery.SchemaField("model", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("station", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("horizon", "INTEGER", mode="NULLABLE"),
    bigquery.SchemaField("weights", "BYTES", mode="REQUIRED"),
    bigquery.SchemaField("n_bytes", "INTEGER", mode="REQUIRED"),
    bigquery.SchemaField("saved_at", "TIMESTAMP", mode="REQUIRED"),
]
val_metric_schema = [
    bigquery.SchemaField("model", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("station", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("horizon", "INTEGER", mode="REQUIRED"),
    bigquery.SchemaField("metric", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("value", "FLOAT", mode="NULLABLE"),
]


def serialize_torch_state(state_dict):
    """Serialize state_dict PyTorch ke bytes (in-memory, tanpa file sementara)."""
    buf = io.BytesIO()
    torch.save(state_dict, buf)
    return buf.getvalue()


saved_at = pd.Timestamp.now(tz="UTC").tz_localize(None)

weight_rows = []
val_metric_rows = []
for station, horizons in tft_models.items():
    for h, model in horizons.items():
        blob = serialize_torch_state(model.state_dict())
        weight_rows.append({
            "model": MODEL_NAME, "station": station, "horizon": int(h),
            "weights": blob, "n_bytes": len(blob), "saved_at": saved_at,
        })
    for hkey, metrics in tft_results[station].items():
        h = int(hkey.replace("h+", ""))
        for metric_name, value in metrics.items():
            val_metric_rows.append({
                "model": MODEL_NAME, "station": station, "horizon": h,
                "metric": metric_name, "value": float(value),
            })

weights_df = pd.DataFrame(weight_rows)
val_metrics_df = pd.DataFrame(val_metric_rows)
total_mb = weights_df["n_bytes"].sum() / 1e6
print(f"Menyiapkan {len(weights_df)} baris bobot ({total_mb:.2f} MB total).")

load_df(weights_df, WEIGHTS_TABLE, weights_schema)
load_df(val_metrics_df, VAL_METRICS_TABLE, val_metric_schema)
print(f"\n\u2714 Bobot + metrik validasi '{MODEL_NAME}' tersimpan di BigQuery.")

## Test Set Evaluation

In [ ]:
def _load_weights_from_bq(station, h):
    query = f"""
        SELECT weights
        FROM `{WEIGHTS_TABLE}`
        WHERE station = @station AND horizon = @horizon
        ORDER BY saved_at DESC
        LIMIT 1
    """
    job_config = bigquery.QueryJobConfig(query_parameters=[
        bigquery.ScalarQueryParameter("station", "STRING", station),
        bigquery.ScalarQueryParameter("horizon", "INT64", h),
    ])
    row = client.query(query, job_config=job_config).to_dataframe()
    if row.empty:
        raise RuntimeError(f"Bobot utk ({station}, h+{h}) tidak ditemukan di {WEIGHTS_TABLE}.")
    return row.iloc[0]["weights"]


def predict_test_tft(target, h):
    test_ds = TMADataset(df_test, target, feature_cols, LOOKBACK, h)
    test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False)

    if target in tft_models and h in tft_models[target]:
        model = tft_models[target][h]
    else:
        # rebuild + load bobot dari BigQuery (mis. notebook baru dibuka ulang)
        model = TFTForecaster(
            n_features=N_FEATURES, hidden_size=TFT_HIDDEN,
            n_heads=TFT_HEADS, dropout=TFT_DROPOUT,
            lstm_layers=TFT_LSTM_LAYERS,
        ).to(DEVICE)
        blob = _load_weights_from_bq(target, h)
        model.load_state_dict(torch.load(io.BytesIO(blob), map_location=DEVICE))

    idx = df_test.index[LOOKBACK + h - 1 : len(df_test)]
    y_pred, y_true = get_predictions(model, test_loader)
    return idx[:len(y_pred)], y_true, y_pred


test_results = {}
test_predictions = []   # baris panjang utk BigQuery

for target in station_cols:
    test_results[target] = {}
    for h in HORIZONS:
        idx, y_true, y_pred = predict_test_tft(target, h)
        metrics = compute_metrics(y_true, y_pred)
        test_results[target][f"h+{h}"] = metrics

        for ts, yt, yp in zip(idx, y_true, y_pred):
            test_predictions.append({
                "model": "tft",
                "station": target,
                "horizon": int(h),
                "target_datetime": pd.Timestamp(ts),
                "y_true": float(yt),
                "y_pred": float(yp),
            })

    row = " | ".join(
        f"h+{h}: NSE={test_results[target][f'h+{h}']['NSE']:.3f}"
        for h in HORIZONS
    )
    print(f"{target:<20} {row}")

test_pred_df = pd.DataFrame(test_predictions)
print(f"\nTest predictions: {test_pred_df.shape[0]:,} rows.")
test_pred_df.head()

In [ ]:
metric_rows = []
for station, horizons in test_results.items():
    for hkey, metrics in horizons.items():
        for metric_name, value in metrics.items():
            metric_rows.append({
                "model": MODEL_NAME, "station": station,
                "horizon": int(hkey.replace("h+", "")),
                "metric": metric_name, "value": float(value),
            })
test_metrics_df = pd.DataFrame(metric_rows)
print(f"Test metrics rows: {len(test_metrics_df)}")

## Write Results to BigQuery

Prediksi dan metrik test-set model ini ditulis ke BigQuery. Autentikasi
mengikuti mekanisme standar Google Cloud (Application Default Credentials,
mis. kredensial bawaan Colab). Mode tulis default `WRITE_TRUNCATE` (ganti
seluruh isi tabel); ubah `WRITE_DISPOSITION` di atas ke `WRITE_APPEND` bila
ingin menumpuk hasil beberapa run.

In [ ]:
pred_schema = [
    bigquery.SchemaField("model", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("station", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("horizon", "INTEGER", mode="REQUIRED"),
    bigquery.SchemaField("target_datetime", "TIMESTAMP", mode="REQUIRED"),
    bigquery.SchemaField("y_true", "FLOAT", mode="NULLABLE"),
    bigquery.SchemaField("y_pred", "FLOAT", mode="NULLABLE"),
]
metric_schema = [
    bigquery.SchemaField("model", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("station", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("horizon", "INTEGER", mode="REQUIRED"),
    bigquery.SchemaField("metric", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("value", "FLOAT", mode="NULLABLE"),
]

# target_datetime -> pastikan tz-naive UTC agar konsisten di BQ
_pred = test_pred_df.copy()
_pred["target_datetime"] = pd.to_datetime(_pred["target_datetime"])

print("Uploading to BigQuery ...")
load_df(_pred, PREDICTIONS_TABLE, pred_schema)
load_df(test_metrics_df, METRICS_TABLE, metric_schema)
print("\n\u2714 BigQuery write selesai.")